# Markov-Modulated Multirisk Common-Shock Reproduction

Source family: Picard, Lefevre and Coulibaly (2003), Loisel
(2004, 2005), Cossette/Landriault/Marceau and related
common-shock multirisk models.

Paper-specific target: reproduce the one-period manual checks
used for the package validation:

- any-line ruin probability: `0.75`;
- total-wealth ruin probability: `0.25`;
- hybrid-region ruin probability: `0.25`;
- positive-dependence impact on any-line ruin: `-0.25`.

Reviewer rubric: please separate comments into mathematical
correctness, API/usability, and teaching value.

In [ ]:
import numpy as np

from ruin_theory import (
    MarkovEnvironment,
    dependence_impact,
    finite_time_markov_modulated_ruin,
)

environment = MarkovEnvironment([1.0], [[1.0]])
increment = [{(0, 0): 0.25, (2, 0): 0.25, (0, 2): 0.25, (2, 2): 0.25}]

common = dict(
    increment_pmfs=increment,
    environment=environment,
    initial_capitals=[1.0, 1.0],
    premiums=[0.0, 0.0],
    horizon=1,
)

any_line = finite_time_markov_modulated_ruin(**common, region="any_line")
total = finite_time_markov_modulated_ruin(**common, region="total")
hybrid = finite_time_markov_modulated_ruin(
    **common,
    region="hybrid",
    severity_limit=1.0,
)

results = {
    "any_line": any_line.ruin_probabilities[-1],
    "total": total.ruin_probabilities[-1],
    "hybrid": hybrid.ruin_probabilities[-1],
}
results

In [ ]:
np.testing.assert_allclose(
    [results["any_line"], results["total"], results["hybrid"]],
    [0.75, 0.25, 0.25],
)

independent = any_line
positive_dependence = finite_time_markov_modulated_ruin(
    [{(0, 0): 0.5, (2, 2): 0.5}],
    environment,
    initial_capitals=[1.0, 1.0],
    premiums=[0.0, 0.0],
    horizon=1,
    region="any_line",
)
impact = dependence_impact(
    independent,
    positive_dependence,
    reference_label="independent",
    comparison_label="positive dependence",
)

assert impact.final_difference == -0.25
impact.final_difference